# 08 — Validasi Integritas Eksperimen

Notebook ini **tidak melatih model untuk ranking**; ia adalah pagar sebelum hasil dibaca. **Jika ada pagar yang gagal, jangan baca ranking — perbaiki pipeline dahulu.**

Yang diperiksa:
1. universum model = 21 (10 ML + 11 DL);
2. data bersih (indeks, NaN, konsistensi fitur stasioner dengan level ber-lag, tanpa lookahead eksogen);
3. fitur train/validasi tidak berubah ketika data uji dirusak;
4. fitness DPformer (MOSSA) tidak dapat melihat uji; early stopping run FINAL (ML & DL) tidak melihat harga periode uji;
5. struktur walk-forward (embargo), skenario B (tanpa lookahead), SVR (ekuivarian terhadap skala target);
6. seluruh hasil utama memakai protokol/revisi/data yang sama dan **sejajar dengan data saat ini** (`y_true` tersimpan == harga data);
7. diagnosis loss curve seluruh 110 run DL: **skill terhadap model nol** + penanda overfit.

**Catatan.** Overfit/underfit tidak bisa "dijamin nol" hanya dari kode. Yang bisa dibuat ketat: mencegah kebocoran, early stopping hanya dari validasi, dan mengukur gejalanya secara jujur tanpa memilih model berdasarkan uji.

In [1]:
import sys, os, json, time, warnings

# Batasi thread CPU supaya laptop tetap responsif selama training berjam-jam.
# Default 2 thread. Ubah lewat env TA_THREADS. Harus di kernel BARU (sebelum numpy/tensorflow aktif).
N_THREADS = int(os.environ.get('TA_THREADS', '2'))
for _v in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS'):
    os.environ.setdefault(_v, str(N_THREADS))

sys.path.insert(0, os.path.join('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from IPython.display import display
try:
    tf.config.threading.set_intra_op_parallelism_threads(N_THREADS)
    tf.config.threading.set_inter_op_parallelism_threads(1)
except RuntimeError:
    print('[PERHATIAN] TensorFlow sudah aktif di kernel ini, batas thread tidak berlaku. Restart kernel.')

from src.config import (OUT, HORIZONS, SETUPS, ML_MODELS, DL_MODELS, ALL_MODELS, LANG, PROTOCOL, ML_REV, QUICK,
                        SEQ_LEN, EPOCHS, PATIENCE, BASE_LR, BASE_WD, DL_BATCH, DL_UNITS)
from src import analysis as A, plots as P, stats as S, runners as R
from src.pipeline import SEED, load_mossa, free_memory
from src.runners import load_data, data_fingerprint

P.set_lang(LANG)
RES = os.path.join(OUT, 'results')
RUNS = os.path.join(RES, 'runs')
FIGS = os.path.join(OUT, 'figures')
MODEL_DIR = os.path.join(RES, 'models')
for _d in (RES, RUNS, FIGS, MODEL_DIR):
    os.makedirs(_d, exist_ok=True)

df, p, exo = load_data('..')
DATA_HASH = data_fingerprint(df)
print(f'Data: {len(p)} hari kerja, {p.index[0].date()} s.d. {p.index[-1].date()} | sidik jari {DATA_HASH}'
      + (' | MODE QUICK (hasil hanya uji kelancaran)' if QUICK else ''))


def simpan(fig, nama):
    P.save_fig(fig, nama, FIGS)
    plt.show()
    plt.close(fig)

from src.config import MOSSA_H, LGB_PARAMS, WF_N_FOLDS, WF_FOLD_LEN
from src.pipeline import (build_lgb_features, train_dl_h, EXO_COLS, EXO_LEVEL_COLS, EXO_LOGRET_COLS, EXO_DIFF_COLS,
                          drift_h, multihorizon_split)
from src.models_dl import BUILDERS
from src import checks as C

assert len(ML_MODELS) == 10 and len(DL_MODELS) == 11 and len(ALL_MODELS) == 21
assert set(ALL_MODELS) == set(ML_MODELS + DL_MODELS)
print('[OK] 21 model = 10 ML + 11 DL')

Data: 4325 hari kerja, 2010-01-05 s.d. 2026-08-03 | sidik jari 7aea8db80d44
[OK] 21 model = 10 ML + 11 DL


## 1. Data bersih dan konsisten

In [2]:
assert p.index.is_monotonic_increasing and p.index.is_unique, 'indeks harus naik & unik'
assert not df.isna().any().any() and np.isfinite(df.to_numpy(float)).all(), 'ada NaN/inf'
assert (p > 0).all(), 'harga <= 0'
gap = pd.Series(p.index[1:] - p.index[:-1]).dt.days
assert gap.max() <= 3, f'celah kalender > 3 hari: {gap.max()}'                 # Jumat -> Senin = 3 hari; libur diisi forward-fill
assert (p.index.dayofweek < 5).all(), 'ada hari akhir pekan'
# fitur stasioner = diff dari level ber-lag-1 (satu-satunya definisi: pipeline.make_exo_features) -> tidak ada lookahead
for c in EXO_LOGRET_COLS:
    assert np.allclose(np.log(df[c]).diff().iloc[1:], df[f'{c}_ret'].iloc[1:], atol=1e-9), c
for c in EXO_DIFF_COLS:
    assert np.allclose(df[c].diff().iloc[1:], df[f'{c}_diff'].iloc[1:], atol=1e-9), c
assert np.allclose(p.pct_change().iloc[1:] * 100, df['emas_ret'].iloc[1:], atol=1e-9)
print(f'[OK] {len(p)} hari kerja, tanpa NaN/duplikat; celah maksimum {gap.max()} hari; fitur stasioner konsisten dengan level ber-lag-1.')

[OK] 4325 hari kerja, tanpa NaN/duplikat; celah maksimum 3 hari; fitur stasioner konsisten dengan level ber-lag-1.


## 2. Pagar kebocoran (merusak data uji lalu memastikan hasil latih/validasi tidak berubah)

In [3]:
# 2a. Fitur tabular: baris train/validasi identik walau data uji dirusak.
print(C.cek_fiturs_test_tidak_mengubah_train_val(p, exo, 20, build_lgb_features))

{'trainval_rows_checked': 3413, 'status': 'OK'}


In [4]:
# 2b. Fitness DPformer (MOSSA) hanya melihat validasi.
guard = C.cek_fitness_tanpa_kebocoran(p, exo, MOSSA_H, train_dl_h, BUILDERS['dpformer'], SEQ_LEN,
                                      DL_UNITS['dpformer'], DL_BATCH['dpformer'])
for k, v in guard.items():
    print(f'{k:30s}: {v:.8f}')
print('[OK] data uji dirusak -> fitness validasi tidak berubah (dan metrik uji berubah: guard sensitif)')


mae_val_asli                  : 18904.91641266
mae_val_test_dirusak          : 18904.91641266
noise_run_ulang               : 0.00000000
mae_test_asli                 : 63137.26514246
mae_test_test_dirusak         : 192339.70401034
[OK] data uji dirusak -> fitness validasi tidak berubah (dan metrik uji berubah: guard sensitif)


In [5]:
# 2c. Early stopping run FINAL (mode uji) juga tidak boleh melihat harga periode uji.
g_ml = C.cek_validasi_final_tanpa_label_test_ml(p, exo, 20, build_lgb_features, LGB_PARAMS)
print('LightGBM :', {k: (round(v, 8) if isinstance(v, float) else v) for k, v in g_ml.items()})
g_dl = C.cek_validasi_final_tanpa_label_test_dl(p, exo, MOSSA_H, train_dl_h, BUILDERS['dpformer'], SEQ_LEN,
                                                DL_UNITS['dpformer'], DL_BATCH['dpformer'])
print('DPformer :', {k: (round(v, 8) if isinstance(v, float) else v) for k, v in g_dl.items()})
print('[OK] model terpilih pada run final tidak berubah ketika harga periode uji dirusak')

Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[8]	valid_0's rmse: 0.0245505
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[8]	valid_0's rmse: 0.0245505
LightGBM : {'best_iter_asli': 8, 'best_iter_test_dirusak': 8, 'rmse_val_asli': 0.02455048, 'rmse_val_test_dirusak': 0.02455048, 'selisih_prediksi_val': 0.0, 'mae_test_asli': 62214.45575355, 'mae_test_test_dirusak': 187682.34021941}
DPformer : {'val_loss_epoch_terakhir_asli': 0.82506734, 'val_loss_epoch_terakhir_test_dirusak': 0.82506734, 'noise_run_ulang': 0.0, 'mae_test_asli': 63137.26514246, 'mae_test_test_dirusak': 192339.70401034}
[OK] model terpilih pada run final tidak berubah ketika harga periode uji dirusak


In [6]:
# 2d. Walk-forward: struktur blok & embargo; satu fold LightGBM dengan data uji dirusak.
from src.config import WF_HORIZONS
for h in sorted(set(HORIZONS) | set(WF_HORIZONS)):
    C.cek_walkforward_embargo(p, h, WF_N_FOLDS, WF_FOLD_LEN)
print(f'[OK] walk-forward {WF_N_FOLDS} fold x {WF_FOLD_LEN} hari: blok kontigu, label train/val < origin uji pertama, untuk h = {sorted(set(HORIZONS) | set(WF_HORIZONS))}')
print(C.cek_walkforward_tanpa_label_test_ml(p, exo, 20, build_lgb_features, fold_idx=0, n_folds=WF_N_FOLDS, fold_len=WF_FOLD_LEN, lgb_params=LGB_PARAMS))

[OK] walk-forward 6 fold x 260 hari: blok kontigu, label train/val < origin uji pertama, untuk h = [1, 5, 10, 20, 30]
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.047873
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[1]	valid_0's rmse: 0.047873
{'best_iter': 1, 'rmse_val': 0.04787302397434792, 'status': 'OK'}


In [7]:
# 2e. Skenario B (eksogen penutupan hari-t): tidak boleh bergantung pada pasar hari-hari berikutnya.
print(C.cek_skenario_b_tanpa_lookahead(df[EXO_LEVEL_COLS]))
# 2f. SVR: prediksi ekuivarian terhadap skala target (mencegah epsilon salah skala seperti versi lama).
print(C.cek_svr_skala_target(p, exo, 5, build_lgb_features))
print('[OK] skenario B tanpa lookahead; SVR ekuivarian terhadap skala target')

{'origin_diperiksa': 2163, 'status': 'OK'}
{'status': 'OK', 'std_prediksi': 0.004340180703044579}
[OK] skenario B tanpa lookahead; SVR ekuivarian terhadap skala target


## 3. Seluruh hasil utama: satu protokol, satu revisi, satu data

`protocol` harus sama (hasil protokol lama = basi); hasil ML harus bertanda `rev = ml-r2`; hasil yang mencatat sidik jari data harus cocok dengan data saat ini; dan `y_true` yang tersimpan harus sama dengan harga di data (menangkap hasil lama dari data yang berbeda).

In [8]:
ml = R.read_runs(RUNS, SETUPS, ML_MODELS, HORIZONS, rev=ML_REV, data_hash=DATA_HASH)
dl = R.read_runs(RUNS, SETUPS, DL_MODELS, HORIZONS, data_hash=DATA_HASH)
expected_ml = len(ML_MODELS) * len(HORIZONS) * len(SETUPS)
expected_dl = len(DL_MODELS) * len(HORIZONS) * len(SETUPS)
print(f'ML valid: {len(ml)} / {expected_ml} | DL valid: {len(dl)} / {expected_dl}')
assert len(ml) == expected_ml, 'ML belum lengkap / revisi basi: jalankan notebook 07'
assert len(dl) == expected_dl, 'DL belum lengkap / protokol basi: jalankan notebook 05 dan 06'
semua = pd.concat([ml, dl], ignore_index=True)
n_tanpa_hash = int(semua['data_hash'].isna().sum()) if 'data_hash' in semua else len(semua)
print(f'Run tanpa catatan sidik jari data (hasil lama, diterima; nilainya diverifikasi di bawah): {n_tanpa_hash} dari {len(semua)}')
cek = C.cek_prediksi_sejajar_data(RUNS, p, SETUPS, ALL_MODELS, HORIZONS)
print(f'[OK] {len(cek)} berkas prediksi: tanggal target = baris uji split utama; y_true == harga data saat ini (selisih maks {cek.maks_selisih_ytrue.max():.1e}); y_pred berhingga')
display(cek.groupby('horizon').n.agg(['min', 'max']).rename(columns={'min': 'n_uji_min', 'max': 'n_uji_maks'}))
# Drift tersimpan == hitung ulang
dr = pd.read_csv(os.path.join(RES, 'drift_benchmark.csv')).set_index('horizon')
for h in HORIZONS:
    m, *_ = drift_h(p, h)
    assert abs(m['MAE'] - dr.loc[h, 'MAE']) <= 1e-6 * max(1, abs(m['MAE'])), f'drift h={h} tidak cocok'
print('[OK] benchmark Drift tersimpan sama dengan hitung ulang')

ML valid: 100 / 100 | DL valid: 110 / 110
Run tanpa catatan sidik jari data (hasil lama, diterima; nilainya diverifikasi di bawah): 0 dari 210
[OK] 210 berkas prediksi: tanggal target = baris uji split utama; y_true == harga data saat ini (selisih maks 0.0e+00); y_pred berhingga


,n_uji_min,n_uji_maks
horizon,,
1,864,864
5,864,864
10,863,863
20,861,861
30,859,859


[OK] benchmark Drift tersimpan sama dengan hitung ulang


## 4. Diagnosis loss curve (110 run DL)

**Mengapa bukan sekadar "train vs validasi".** Target DL di-z-score pada train, sehingga *model nol* (selalu memprediksi rata-rata target train) memiliki MSE = 1 di train dan `val_null` di validasi (≠ 1 karena pergeseran rezim). Dari situ dua atribut **terpisah**:

| Atribut | Arti |
|---|---|
| **kelas** (skill epoch-terbaik terhadap model nol di validasi, toleransi ±2%) | `BELAJAR` (lebih baik dari model nol) · `TANPA_SINYAL` (≈ model nol: hanya menemukan rata-rata) · `DI_BAWAH_NOL` (epoch terbaik pun lebih buruk dari model nol) |
| **overfit** (boolean) | setelah epoch terbaik, loss validasi naik > 5% dari `val_null` sementara loss train masih turun > 3% |

Bobot yang dipakai untuk uji adalah epoch terbaik (`restore_best_weights`), jadi *overfit* menjelaskan **bentuk kurva**, bukan model akhir yang rusak.

In [9]:
diag = A.loss_diagnosis_table(RUNS, p, SETUPS, DL_MODELS, HORIZONS, SEQ_LEN)
diag.to_csv(os.path.join(RES, 'diagnosis_loss.csv'), index=False)
print(f'{len(diag)} history didiagnosis (diharapkan {len(DL_MODELS) * len(HORIZONS) * len(SETUPS)})')
assert len(diag) == len(DL_MODELS) * len(HORIZONS) * len(SETUPS)
print('\nJumlah run per kelas:')
display(diag.groupby(['setup', 'kelas']).size().unstack(fill_value=0))
print('Jumlah run dengan overfit setelah epoch terbaik:')
display(diag.groupby('setup').overfit.agg(['sum', 'count']).rename(columns={'sum': 'overfit', 'count': 'total'}))
print('Median epoch terbaik per model (semua setup & horizon):')
display(diag.groupby(['setup', 'model']).best_epoch.median().unstack(0))

110 history didiagnosis (diharapkan 110)

Jumlah run per kelas:


kelas,BELAJAR,DI_BAWAH_NOL,TANPA_SINYAL
setup,,,
multivariat,4,26,25
univariat,0,12,43


Jumlah run dengan overfit setelah epoch terbaik:


,overfit,total
setup,,
multivariat,33,55
univariat,6,55


Median epoch terbaik per model (semua setup & horizon):


setup,multivariat,univariat
model,,
bigru,2.0,1.0
bilstm,2.0,2.0
dlinear,62.0,49.0
dpformer,7.0,6.0
gru,2.0,1.0
gru_att,1.0,2.0
lstm,2.0,2.0
lstm_att,1.0,1.0
nlinear,69.0,67.0


In [10]:
ringkas = (diag.groupby(['setup', 'model'])
           .agg(skill_val_median=('skill_val_best', 'median'), n_belajar=('kelas', lambda x: int((x == 'BELAJAR').sum())),
                n_tanpa_sinyal=('kelas', lambda x: int((x == 'TANPA_SINYAL').sum())),
                n_di_bawah_nol=('kelas', lambda x: int((x == 'DI_BAWAH_NOL').sum())),
                n_overfit=('overfit', 'sum'), epoch_terbaik_median=('best_epoch', 'median')).round(3).reset_index())
ringkas.to_csv(os.path.join(RES, 'diagnosis_loss_ringkas.csv'), index=False)
ringkas.sort_values(['setup', 'skill_val_median'], ascending=[True, False])

,setup,model,skill_val_median,n_belajar,n_tanpa_sinyal,n_di_bawah_nol,n_overfit,epoch_terbaik_median
3,multivariat,dpformer,0.010,1,4,0,2,7.0
6,multivariat,lstm,0.004,1,3,1,3,2.0
1,multivariat,bilstm,0.002,1,3,1,2,2.0
5,multivariat,gru_att,-0.008,1,2,2,3,1.0
7,multivariat,lstm_att,-0.008,0,5,0,4,1.0
4,multivariat,gru,-0.019,0,4,1,5,2.0
0,multivariat,bigru,-0.025,0,2,3,5,2.0
2,multivariat,dlinear,-0.052,0,0,5,0,62.0
9,multivariat,patchtst,-0.068,0,0,5,5,2.0
10,multivariat,transformer,-0.098,0,2,3,4,1.0


### Cara membaca (dan apa yang *tidak* bisa diperbaiki)

- **Kurva "semua bagus" tidak bisa dipaksakan tanpa mengubah data.** Return emas harian hampir efisien: sinyal yang dapat dipelajari dari harga/makro lampau sangat kecil. Kurva yang datar mendekati model nol (`TANPA_SINYAL`) adalah perilaku **yang benar**, bukan bug. Kurva yang turun jauh di train tetapi validasinya naik (`overfit`) menandakan model menghafal; early stopping sudah menjaganya dengan memulihkan epoch terbaik.
- **Mengapa loss validasi bisa lebih rendah dari loss train:** validasi (≈2021–2023) adalah periode sideways tenang, sedangkan train memuat periode yang lebih bergejolak/basi (notebook 04). Itu artefak rezim, bukan kebocoran — dibuktikan oleh pagar bagian 2.
- **`DI_BAWAH_NOL` pada model multivariat** berarti rata-rata target train tidak cocok dengan validasi (pergeseran rezim) dan/atau model cepat menghafal fitur makro yang berisik. Ini temuan yang dilaporkan, bukan alasan untuk mengutak-atik uji.
- Jika ada model yang ingin dituning ulang, gunakan **validasi saja** (bukan uji), dan laporkan konfigurasinya.
- Figur kurva loss lengkap per (setup, horizon) ada di notebook 12.